# AML/KYC Compliance Flagging Agent — Notebook Walkthrough**HBF2212 — Artificial Intelligence in Finance, Project 2**This notebook demonstrates the AML/KYC compliance agent's core pipelinestep by step, outside the Streamlit interface. It clones the project'spublic GitHub repository and runs the same `rules_engine.py` module usedby the deployed app, so the logic shown here is identical to production.- **Live app:** https://aml-kyc-agent-ryut5cxe9.streamlit.app  *(replace with your final URL)*- **GitHub repo:** https://github.com/mjt040305-cloud/aml-kyc-agent**Pipeline stages covered in this notebook:**1. Read — load transaction data2. Analyse — run the AML rules engine (4 risk categories, 6 rules)3. Decide/flag — risk scoring and bucketing4. Human oversight checkpoint — simulated compliance officer review5. Output — final compliance reportThe full deployed app additionally orchestrates this pipeline as a**LangGraph state graph** (`agent_graph.py`) with a genuine `interrupt()`pause at the human oversight step — see the final section of thisnotebook, and the live app itself, for that orchestration layer inaction.

## 1. Setup — clone the repository and install dependencies

In [ ]:
!git clone -q https://github.com/mjt040305-cloud/aml-kyc-agent.git%cd aml-kyc-agent!pip install -q -r requirements.txt

## 2. Read — load transaction dataWe use the bundled synthetic dataset (`sample_transactions.csv`), which contains normal activity plus deliberately embedded suspicious patterns for testing.

In [ ]:
import pandas as pdfrom rules_engine import analyse_transactions, DEFAULT_CONFIG, SEVERITY_ICON, CATEGORIESdf = pd.read_csv("sample_transactions.csv")print(f"Loaded {len(df)} transactions")df.head(10)

## 3. Analyse + Decide/Flag — run the AML rules engineEach transaction is scored across four risk categories (Customer, Transaction, Geographic, Behavioural) and bucketed into None / Low / Medium / High risk. `DEFAULT_CONFIG` holds the AML thresholds — the same dict a compliance officer can override from the deployed app's sidebar.

In [ ]:
print("Default AML rule configuration:")for k, v in DEFAULT_CONFIG.items():    print(f"  {k}: {v}")analysed = analyse_transactions(df)analysed[["transaction_id", "customer_id", "amount", "risk_score", "risk_bucket"]].head(15)

In [ ]:
summary = analysed["risk_bucket"].value_counts()print("Risk distribution:")print(summary)

### Inspect the risk breakdown for the highest-scoring transactionThis shows the explainability that makes the rule-based approach auditable: every category contribution and every triggered rule is visible, not a black-box score.

In [ ]:
top = analysed.iloc[0]print(f"Transaction {top['transaction_id']} — Customer {top['customer_id']} — ${top['amount']:,.2f}")print(f"Overall risk score: {top['risk_score']} ({top['risk_bucket']})")print()print("Category breakdown:")for cat, score in top["category_scores"].items():    print(f"  {cat}: {score}")print()print("AML rules triggered:")for rule in top["triggered_rules"]:    icon = SEVERITY_ICON[rule["severity"]]    print(f"  {icon} {rule['label']} ({rule['category']}) — {rule['reason']}")

## 4. Human Oversight Checkpoint (simulated)In the deployed Streamlit app, this step is a genuine LangGraph `interrupt()` — the agent pauses and cannot proceed until a compliance officer records a decision. In this notebook we simulate that same decision-making step programmatically so the pipeline can run end-to-end without an interactive UI.

In [ ]:
pending = analysed[analysed["risk_bucket"].isin(["High", "Medium"])].copy()print(f"{len(pending)} transaction(s) require human review before this pipeline can produce final output.")pending[["transaction_id", "customer_id", "amount", "risk_bucket", "flag_reasons"]]

In [ ]:
# Simulated compliance officer decisions.# In the live app, a human enters these through the UI and the LangGraph# agent literally cannot proceed past this point without them.import randomrandom.seed(1)decisions = {}for _, row in pending.iterrows():    decision = "Escalate to SAR filing" if row["risk_bucket"] == "High" else "Approve (false positive)"    decisions[row["transaction_id"]] = {        "status": decision,        "reviewer": "J. Musina (compliance officer)",        "notes": "Reviewed against customer KYC file and transaction history.",    }for txn_id, d in decisions.items():    print(f"{txn_id}: {d['status']}")

## 5. Output — final compliance report

In [ ]:
analysed["review_status"] = analysed["transaction_id"].map(    lambda tid: decisions.get(tid, {}).get("status", "Not required"))analysed["reviewed_by"] = analysed["transaction_id"].map(    lambda tid: decisions.get(tid, {}).get("reviewer", ""))analysed["reviewer_notes"] = analysed["transaction_id"].map(    lambda tid: decisions.get(tid, {}).get("notes", ""))final_report = analysed[[    "transaction_id", "customer_id", "amount", "date", "risk_bucket",    "risk_score", "flag_reasons", "review_status", "reviewed_by", "reviewer_notes"]]final_report.to_csv("notebook_compliance_report.csv", index=False)print("Saved notebook_compliance_report.csv")final_report.head(15)

## 6. Agent orchestration (LangGraph) — see the deployed app for live executionThe cells above call `rules_engine.analyse_transactions()` directly forclarity. The deployed Streamlit app instead runs this same function*inside* a LangGraph `StateGraph` (`agent_graph.py`):```START -> analyse -> human_review (interrupt) -> output -> END```The `human_review` node calls LangGraph's `interrupt()`, which genuinelypauses graph execution — checkpointed via `MemorySaver` — until a`Command(resume=<reviews>)` is supplied. This cannot be demonstrated in astatic notebook (it requires an interactive UI to supply the resumecommand), which is why the live Streamlit app is the primary place to seethe human oversight checkpoint enforced structurally rather thansimulated as it is in Section 4 above.**To see it live:** open the deployed app, load the sample data, run theanalysis, and note that the app will not display Step 5 output untilevery flagged transaction has a recorded decision — the agent graph isgenuinely paused, not just waiting on a disabled button.